## Probability and Statistics (MT-2005) Spring 2026
## Project 3

This project contains 10 questions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
"""
Instructions:
All answers must be written in a text markdown at the end of colab file with proper labels. Additionally, the naming convention specified must be followed as well.
Not doing so would result in a deduction of points; and your project cannot be claimed for rechecking.

Naming convention: section_rollNumber
example: G_i230030
If you fail to follow the naming convention or the submission guidelines, your project will not be graded and cannot be claimed for rechecking.
"""

## Preprocessing and basic exploration

In [ ]:
#importing data
df = pd.read_csv("GlobalTerrorismDatabase.csv", encoding='latin1', on_bad_lines='skip', engine='python')

In [ ]:
print(df.shape) #print the rows and columns
print("\nDate range:", df['iyear'].min(), "-", df['iyear'].max())

Selection of Data for Hidden Markov Models

In [ ]:
# Select columns relevant for HMM analysis
relevant_cols = [
    'iyear',      # Year
    'imonth',     # Month (1-12)
    'country_txt', # Country name
    'nkill',      # Number killed
    'nwound',     # Number wounded
    'attacktype1_txt', # Type of attack
    'weaptype1_txt',   # Weapon type
    'targtype1_txt',   # Target type
    'success'     # Was attack successful?
]

# Create a working dataframe
df_hmm = df[relevant_cols].copy()
print("Working dataset shape:", df_hmm.shape)
print("\nMissing value counts:")
print(df_hmm.isnull().sum())

Q1. Explain why is it more appropriate to fill in the missing values with median instead of the mean for this particular dataset.

In [ ]:
#filling missing values
#numeric columns are filled with median instead of the mean.
numeric_cols = ['nkill', 'nwound', 'success']
for col in numeric_cols:
    df_hmm[col] = df_hmm[col].fillna(df_hmm[col].median())

# For categorical columns: fill with mode
categorical_cols = ['attacktype1_txt', 'weaptype1_txt', 'targtype1_txt']
for col in categorical_cols:
    df_hmm[col] = df_hmm[col].fillna(df_hmm[col].mode()[0] if len(df_hmm[col].mode()) > 0 else 'Unknown')

# Check that missing values are handled
print("\nMissing values after imputation:")
print(df_hmm.isnull().sum())

In [ ]:
def remove_outliers_iqr(df, column, multiplier=3):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    print(f"Removing {len(outliers)} outliers from {column} ({(len(outliers)/len(df))*100:.2f}%)")
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

df_hmm = remove_outliers_iqr(df_hmm, 'nkill', multiplier=3)
df_hmm = remove_outliers_iqr(df_hmm, 'nwound', multiplier=3)
print(f"\nDataset after outlier removal: {len(df_hmm)} attacks") #This is the number of rows you have to wrestle with now.

Now we create a monthly time series

In [ ]:
# Create a monthly time series (all months from 1970 to 2020)
df_hmm['date'] = pd.to_datetime(df_hmm['iyear'].astype(str) + '-' + df_hmm['imonth'].astype(str) + '-01', errors='coerce')
df_hmm = df_hmm.dropna(subset=['date'])
df_hmm = df_hmm.sort_values('date')
date_range = pd.date_range(start='1970-01-01', end='2020-12-01', freq='MS')
monthly_data = pd.DataFrame({'date': date_range})
monthly_attacks = df_hmm.groupby('date').agg({
    'nkill': ['sum', 'mean', 'count'],
    'nwound': 'sum',
    'success': 'mean'
}).round(2)

monthly_attacks.columns = ['total_killed', 'avg_killed_per_attack', 'attack_count', 'total_wounded', 'success_rate']
monthly_attacks = monthly_attacks.reset_index()
monthly_attacks = monthly_attacks.fillna(0)

Hidden Markov Models require discrete observations. So in this markdown, we discretize the observations.

In [ ]:
# Create discretized observation (0, 1, 2)
"""Convert attack count to discrete state:
0 = Low activity (0 attacks)
1 = Medium activity (1-10 attacks)
2 = High activity (11+ attacks)
"""
def discretize_attack_intensity(attack_count):
    if attack_count == 0:
        return 0
    elif attack_count <= 10:
        return 1
    else:
        return 2

def discretize_casualty_severity(avg_killed):
    """Convert casualties to discrete state"""
    if avg_killed == 0:
        return 0
    elif avg_killed <= 2:
        return 1
    else:
        return 2

monthly_attacks['intensity_obs'] = monthly_attacks['attack_count'].apply(discretize_attack_intensity)
monthly_attacks['casualty_obs'] = monthly_attacks['avg_killed_per_attack'].apply(discretize_casualty_severity)
monthly_attacks['combined_obs'] = monthly_attacks['intensity_obs'] * 3 + monthly_attacks['casualty_obs']

# Let's use intensity_obs as our primary observation sequence
observation_sequence = monthly_attacks['intensity_obs'].values

### Hidden Markov Models

Forward Algorithm

Q2. Generate all possible arrangements of the observation_sequence and compute the likelihood probability for each of them using the forward algorithm.

In [ ]:
import numpy as np
N = 3 # Number of hidden states
M = 3 # Number of observation symbols (0, 1, 2)
pi = np.array([0.3, 0.4, 0.3]) # Initial state probabilities

# Transition probability matrix (N x N)
A = np.array([
[0.8, 0.15, 0.05],
 [0.2, 0.6, 0.2],
 [0.1, 0.2, 0.7]
])

# Emission probability matrix (N x M)
B = np.array([
[0.7, 0.2, 0.1],
 [0.2, 0.6, 0.2],
 [0.1, 0.3, 0.6]
])

# Implementation of the forward algorithm
def forward_algorithm(obs, pi, A, B):
    T = len(obs)
    N = len(pi)
    alpha = np.zeros((T, N))
    # Initialization
    alpha[0] = pi * B[:, obs[0]]

    # Recursion
    for t in range(1, T):
        for j in range(N):
            alpha[t, j] = np.sum(alpha[t - 1] * A[:, j]) * B[j, obs[t]]

    # Termination
    likelihood = np.sum(alpha[T - 1])
    return alpha, likelihood

observation_sequence = [0, 1, 2] #Question 2 <<<-------

alpha, likelihood = forward_algorithm(observation_sequence, pi, A, B)
print("Likelihood of observation sequence:", likelihood)

Viterbi Algorithm

Q3. What is a degenerate model? How does it kill the purpose of HMM? How can we change our current model into a degenrate model?

Q4. When can the model collapse into a single dominant state? (All the observed states are the same). Discuss how the use of log probabilities may be able to solve this problem. Does data skewness have to do anything with a single dominant state?

Q5. Use all the possible combinations of the observation sequence and compute the most likely hidden states.


In [ ]:
# Implementation
def viterbi_algorithm(obs, pi, A, B):
    T = len(obs)
    N = len(pi)

    delta = np.zeros((T, N))
    psi = np.zeros((T, N), dtype=int)

    # Initialization
    delta[0] = pi * B[:, obs[0]]

    # Recursion
    for t in range(1, T):
        for j in range(N):
            prob = delta[t-1] * A[:, j]
            psi[t, j] = np.argmax(prob)
            delta[t, j] = np.max(prob) * B[j, obs[t]]

    # Backtracking
    states = np.zeros(T, dtype=int)
    states[T-1] = np.argmax(delta[T-1])

    for t in range(T-2, -1, -1):
        states[t] = psi[t+1, states[t+1]]
    best_path_prob = np.max(delta[T-1])
    return states, best_path_prob

most_likely_states, bpp = viterbi_algorithm(observation_sequence, pi, A, B)
print("Most likely hidden states:", most_likely_states)
print("Probability of best path: ", bpp)

Baum Welch

Q6. Tweak the maximum iterations, and make observations on the values of Pi, A, and B

Q7. Give an interpretation on Pi

Q8.Looking at A, how can you tell that the states can transition between one another?

Q9. Why might Baum–Welch assign one hidden state almost exclusively to “no attacks”?

Q10. What real-world interpretation can be assigned to each hidden state?

In [ ]:
def baum_welch(obs, N, M, max_iter=20):
    obs = np.array(obs)
    T = len(obs)
    # 1. Initialization
    pi = np.full(N, 1 / N)

    A = np.random.rand(N, N)
    A /= A.sum(axis=1, keepdims=True)

    B = np.random.rand(N, M)
    B /= B.sum(axis=1, keepdims=True)

    for _ in range(max_iter):
        # 2. Forward
        alpha = np.zeros((T, N))
        alpha[0] = pi * B[:, obs[0]]

        for t in range(1, T):
            for j in range(N):
                alpha[t, j] = np.sum(alpha[t - 1] * A[:, j]) * B[j, obs[t]]

        # normalize
        alpha /= np.sum(alpha, axis=1, keepdims=True)

        # 3. Backward
        beta = np.zeros((T, N))
        beta[-1] = 1

        for t in range(T - 2, -1, -1):
            for i in range(N):
                beta[t, i] = np.sum(A[i] * B[:, obs[t + 1]] * beta[t + 1])

        # normalize
        beta /= np.sum(beta, axis=1, keepdims=True)

        # 4. Gamma (state prob)
        gamma = alpha * beta
        gamma /= np.sum(gamma, axis=1, keepdims=True)

        # 5. Update pi
        pi = gamma[0]

        # 6. Update A (transition)
        for i in range(N):
            for j in range(N):
                num = 0.0
                den = 0.0
                for t in range(T - 1):
                    num += gamma[t, i] * A[i, j]
                    den += gamma[t, i]
                A[i, j] = num / (den + 1e-12)

        # normalize rows
        A /= A.sum(axis=1, keepdims=True)

        # 7. Update B (emission)
        for j in range(N):
            for k in range(M):
                num = 0.0
                den = 0.0
                for t in range(T):
                    if obs[t] == k:
                        num += gamma[t, j]
                    den += gamma[t, j]
                B[j, k] = num / (den + 1e-12)

        # normalize rows
        B /= B.sum(axis=1, keepdims=True)

    return pi, A, B
pi_learned, A_learned, B_learned = baum_welch(
    observation_sequence,
    N=3,
    M=3,
    max_iter=500)

print("Pi:\n", pi_learned)
print("A:\n", A_learned)
print("B:\n", B_learned)